#### Vector search by hand

In [ ]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
#cosine similarity
v1.dot(dv)

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
#join question and answer together for further encoding
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [ ]:
#encode documents by batches
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
#conver into 2D matrix
import numpy as np
X = np.array(vectors)

In [ ]:
scores = X.dot(v1)
idx = np.argmax(scores)
idx, scores[idx]

In [ ]:
documents[553]

In [ ]:
top5 = np.argsort(-scores)[:5]

In [ ]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

#### Vector serach with minsearch

In [ ]:
from minsearch import VectorSearch

# pass the numpy array X with all embeddings and the list of documents as payload
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

# It computes the dot product between each vector (after filtering) and query vector.
results = vindex.search(query_vector, num_results=5)

In [ ]:
# filter by keyword fields
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

#### RAG with Vector Search

Uses minsearch for vector search. <br>
Problems:

It rebuilds the index on every startup <br>
It keeps everything in memory <br>
It searches by brute force

In [ ]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [ ]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)

#### Vector Search with sqlitesearch

persistent equivalent of minsearch <br>
sqlitesearch supports three ANN modes: <br>

lsh (default): up to 100K vectors, random hyperplane projections <br>
ivf: 10K-500K vectors, K-means clustering <br>
hnsw: 10K-1M+ vectors, proximity graph (highest recall)<br>
All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [ ]:
# Indexing the data
vs_index.fit(vectors, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
# Filtering by course
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [ ]:
# Closing the connection
vs_index.close()